# ERCOT Houston Hub — EDA & Volatility Modeling Plan

This notebook covers:
1. **Setup** — load processed data, define constants
2. **Target EDA** — distribution and time-series structure of RTM price volatility
3. **Feature EDA** — key predictors (net load, wind error, forecast uncertainty, ancillary prices)
4. **Regime analysis** — extreme events (Winter Storm Uri, summer peaks, autocorrelation)
5. **Modeling roadmap** — recommended models, features, and validation strategy

**Data range:** 2017-07-01 → 2025-12-31 (hourly, ~74,000 rows).  
**2026 data is held out as out-of-sample test set — do not load it here.**

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

PROCESSED_ROOT = Path('data/processed/ercot')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
sns.set_style('whitegrid')

# Spike threshold — adjust after EDA below
SPIKE_THRESHOLD = 500   # $/MWh

print('Imports OK')

In [ ]:
df = pd.read_parquet(PROCESSED_ROOT / 'ercot_combined.parquet')
df['ts_utc'] = pd.to_datetime(df['ts_utc'])
df = df.sort_values('ts_utc').reset_index(drop=True)

print(f'Rows: {len(df):,}  |  Columns: {len(df.columns)}')
print(f'Range: {df.ts_utc.min()} -> {df.ts_utc.max()}')
print(f'\nColumns:\n{list(df.columns)}')

## Data Design: Two Separate Jobs

### `ercot_combined` — delivery-aligned actuals (EDA only)

`ercot_combined.parquet` aligns every dataset by **delivery time** (`ts_utc`). Each row contains the actual realized values for that delivery hour, regardless of when ERCOT published them. This is the right table for EDA: correlations, distributions, seasonal patterns, regime analysis.

**Do not use `ercot_combined` directly as model input features** — it contains data that was not yet published at prediction time (6PM D-1).

---

### Data availability at the 6PM D-1 prediction cutoff

When predicting delivery day **D** at **6PM on D-1**, the following data is actually available:

| Column group in `ercot_combined` | Published | Available at 6PM D-1? | Safe lag to use |
|---|---|---|---|
| `load_houston`, `load_total`, `wz_*` | D+1 05:50 UTC | **No** — D-1 actuals not out yet | D-2 load (use individual parquet) |
| `dam_price_*`, `system_lambda`, `mcpc_*` | D-1 12:32 UTC | **Yes** — for delivery day D | Same-day D (no lag needed) |
| `rtm_price_*` | Real-time per interval | Partial D-1 (up to ~17:45), full D-2 | D-2 for complete day |
| `total_resource_mw`, `total_irr_mw` (outage) | Hourly rolling | **Yes** — latest revision available | Most recent posting |
| `fc_*`, `wf_*`, `genf_*` (forecasts) | Built with 6PM cutoff | **Yes** — already filtered at save time | Direct use |
| `wind_error_system`, `wind_error_houston` | D+1 (actuals) | **No** | Use as lags (D-2 and earlier) |

---

### For model training: use individual parquets + `post_datetime` filter

Each individual parquet (e.g., `np6_346_houston.parquet`) contains a `post_datetime` column — the exact time ERCOT published that row. The correct pattern for building a leakage-safe feature matrix is:

```python
cutoff = pd.Timestamp('2024-12-15 18:00')   # 6PM D-1

# Load actuals — D-2 (post_datetime filter drops D-1 automatically)
load = pd.read_parquet(PROCESSED_ROOT / 'np6_346_houston.parquet')
load_avail = load[load['post_datetime'] <= cutoff]
# most recent complete day: ts_utc.dt.date == delivery_date - 2 days

# DAM prices — D (fully available, posted ~noon D-1)
dam = pd.read_parquet(PROCESSED_ROOT / 'np4_190_dam_houston.parquet')
dam_avail = dam[dam['post_datetime'] <= cutoff]

# Forecasts — already safe (built with 6PM cutoff at processing time)
fc = pd.read_parquet(PROCESSED_ROOT / 'np3_565_forecast1.parquet')
```

The `post_datetime` filter enforces availability mechanically — using the wrong lag produces an empty result (caught immediately) rather than silent leakage.

**Forecast parquets** (`np3_565_forecast1`, `np4_732_forecast1`, `np3_233_forecast1`) do **not** have a `post_datetime` column — they already encode the 6PM D-1 cutoff via `horizon_h` and can be used directly.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

_PROC = Path('data/processed/ercot')

def build_features(delivery_date, _cache=None):
    """
    Build a leakage-safe (24, n_features) feature matrix for a given delivery date.
    Uses only data available at 6PM D-1 cutoff.

    Parameters
    ----------
    delivery_date : pd.Timestamp or date
    _cache : dict, optional
        Pre-loaded parquet DataFrames keyed by dataset name (e.g. 'np4_190_dam_houston').
        Pass this when looping over many dates for a large speedup.

    Returns
    -------
    pd.DataFrame, shape (24, n_features), indexed by ts_utc (delivery hours)
    """
    delivery_date = pd.Timestamp(delivery_date)
    cutoff = delivery_date - pd.Timedelta(days=1) + pd.Timedelta(hours=18)  # 6PM D-1
    d2_date = (delivery_date - pd.Timedelta(days=2)).date()

    def _load(name):
        if _cache is not None:
            return _cache[name]
        return pd.read_parquet(_PROC / f'{name}.parquet')

    def _avail(df):
        return df[df['post_datetime'] <= cutoff]

    def _day(df, date):
        return df[df['ts_utc'].dt.date == date]

    # --- DAM prices (published D-1 ~12:32 UTC — same-day D available) ---
    dam = _day(_avail(_load('np4_190_dam_houston')), delivery_date.date())[['ts_utc', 'dam_price_houston']]

    # --- System lambda (same as DAM) ---
    lam = _day(_avail(_load('np4_523_system_lambda')), delivery_date.date())[['ts_utc', 'system_lambda']]

    # --- Ancillary MCPC (same as DAM) ---
    anc_cols = ['mcpc_ecrs', 'mcpc_regup', 'mcpc_rrs', 'mcpc_nspin', 'mcpc_regdn']
    anc_dfs = []
    for col in anc_cols:
        a = _day(_avail(_load(f'np4_188_{col}')), delivery_date.date())[['ts_utc', col]]
        anc_dfs.append(a)

    # --- Load actuals D-2 (published D+1 — use D-2, align to delivery hour) ---
    load_raw = _day(_avail(_load('np6_346_houston')), d2_date)[['ts_utc', 'houston']].copy()
    load_raw['ts_utc'] = load_raw['ts_utc'] + pd.Timedelta(days=2)
    load_raw = load_raw.rename(columns={'houston': 'load_houston_d2'})

    # --- RTM prices D-2 (real-time, use D-2 for complete day) ---
    rtm_raw = _day(_avail(_load('np6_905_rtm_hourly_houston')), d2_date)[['ts_utc', 'rtm_price_mean', 'rtm_price_std']].copy()
    rtm_raw['ts_utc'] = rtm_raw['ts_utc'] + pd.Timedelta(days=2)
    rtm_raw = rtm_raw.rename(columns={'rtm_price_mean': 'rtm_mean_lag48', 'rtm_price_std': 'rtm_std_lag48'})

    # --- Outage capacity (most recent posting <= cutoff) ---
    out = _day(_avail(_load('np3_233_outage_total')), delivery_date.date())[['ts_utc', 'total_resource_mw']]
    if out.empty:
        out_all = _avail(_load('np3_233_outage_total'))
        latest_val = out_all.sort_values('post_datetime').iloc[-1]['total_resource_mw']
        out = pd.DataFrame({'ts_utc': pd.date_range(delivery_date, periods=24, freq='h'),
                            'total_resource_mw': latest_val})

    # --- Wind system actuals D-2 ---
    wind_raw = _day(_avail(_load('np4_732_wind_system')), d2_date)[['ts_utc', 'wgrpp_system_wide', 'wind_error_system']].copy()
    wind_raw['ts_utc'] = wind_raw['ts_utc'] + pd.Timedelta(days=2)

    # --- Load forecast (6PM D-1 cutoff already baked in via horizon_h) ---
    fc = _day(_load('np3_565_forecast1'), delivery_date.date())[['ts_utc', 'fc_system_total', 'fc_coast']]

    # --- Wind forecast (same) ---
    wfc = _day(_load('np4_732_forecast1'), delivery_date.date())[['ts_utc', 'wf_stwpf_system_wide']]

    # --- Weather (no publication lag — delivery-day actual used as proxy for day-ahead forecast) ---
    # Exclude pressure_hpa_texas (elevation artifact — West TX stations at high altitude)
    wx_cols = ['ts_utc', 'temp_f_houston_avg', 'humidity_pct_houston_avg',
               'wind_gust_mph_houston_avg', 'precip_in_houston_avg', 'temp_f_texas_avg']
    wx = _day(_load('weather_hourly'), delivery_date.date())[wx_cols]

    # --- Merge all onto delivery-day ts_utc spine ---
    spine = pd.DataFrame({'ts_utc': pd.date_range(delivery_date, periods=24, freq='h')})
    feats = spine.copy()
    for d in [dam, lam, load_raw, rtm_raw, out, wind_raw, fc, wfc, wx] + anc_dfs:
        feats = feats.merge(d, on='ts_utc', how='left')

    # --- Engineered features ---
    feats['ecrs_available']  = (feats['mcpc_ecrs'].notna() & (feats['mcpc_ecrs'] > 0)).astype(int)
    feats['log_mcpc_ecrs']   = np.log1p(feats['mcpc_ecrs'].fillna(0))
    feats['ecrs_above_50']   = (feats['mcpc_ecrs'].fillna(0) > 50).astype(int)
    feats['mcpc_ecrs']       = feats['mcpc_ecrs'].fillna(0)  # pre-2021: ECRS not yet procured
    feats['hour']            = feats['ts_utc'].dt.hour
    feats['month']           = feats['ts_utc'].dt.month
    feats['dow']             = feats['ts_utc'].dt.dayofweek

    return feats.set_index('ts_utc')


# --- Quick test: build features for one delivery date ---
test_date = pd.Timestamp('2024-06-15')
feats = build_features(test_date)
print(f"Feature matrix shape: {feats.shape}")
print(f"Columns ({len(feats.columns)}): {list(feats.columns)}")
print(f"\nSample (first 3 rows):")
print(feats.head(3).to_string())
print(f"\nNull counts:")
print(feats.isnull().sum()[feats.isnull().sum() > 0].to_string() or "  None")

## Feature Dictionary

All features are constructed to be available at the **6PM D-1 prediction cutoff** (no leakage).
The table below explains each of the 27 features used in the model.

---

### Market Price Features

| Feature | Source | Availability | Why it matters |
|---|---|---|---|
| `dam_price_houston` | NP4-190 DAM settlement | D-1 ~noon UTC | Day-ahead market price for HB_HOUSTON hub. Captures the market's consensus expectation of delivery-day value. High DAM prices signal tight supply and tend to precede RTM volatility. |
| `system_lambda` | NP4-523 DAM system lambda | D-1 ~noon UTC | Marginal cost of energy at DAM clearing. Closely related to DAM price but at the system level — divergence from hub price indicates congestion. |
| `rtm_mean_lag48` | NP6-905 RTM (D-2 actuals) | D-2 complete | 48-hour lag of realized RTM price mean. Captures persistence of price regime — high prices on D-2 suggest demand/supply tightness that may carry forward. |
| `rtm_std_lag48` | NP6-905 RTM (D-2 actuals) | D-2 complete | 48-hour lag of realized RTM price volatility (std of 15-min intervals within hour). Directly measures recent volatility — strong autocorrelation in volatility (GARCH effect) makes this one of the most predictive features. |

---

### Ancillary Service Prices

These reflect the DAM-cleared prices for reserve products. High prices signal that the system operator sees limited operating reserves — a leading indicator of real-time scarcity.

| Feature | Source | Why it matters |
|---|---|---|
| `mcpc_regup` | NP4-188 MCPC Reg-Up | Regulation-up reserve scarcity → RTM price spikes when committed capacity is tight |
| `mcpc_regdn` | NP4-188 MCPC Reg-Down | Regulation-down reserve → indicates oversupply risk (negative price events) |
| `mcpc_rrs` | NP4-188 MCPC RRS | Responsive reserve service — proxy for spinning reserve margin |
| `mcpc_nspin` | NP4-188 MCPC Non-Spin | Non-spinning reserve — elevated prices suggest offline backup capacity is scarce |
| `mcpc_ecrs` | NP4-188 MCPC ECRS | ERCOT Contingency Reserve Service (launched ~2021). High ECRS prices indicate the system is operating near its reliability limit. Pre-2021 set to 0. |
| `log_mcpc_ecrs` | Engineered | Log-transform of ECRS (heavy right tail). Compresses extreme scarcity events. |
| `ecrs_available` | Engineered | Binary: 1 if ECRS product existed (post-2021) and price > 0. Prevents the model from treating pre-2021 zeros the same as genuine zero-price ECRS. |
| `ecrs_above_50` | Engineered | Binary: 1 if `mcpc_ecrs` > $50. Flags extreme reserve scarcity events. |

---

### Load & Supply Features

| Feature | Source | Availability | Why it matters |
|---|---|---|---|
| `load_houston_d2` | NP6-346 (D-2 actuals) | D-2 complete | Realized Houston zone load two days prior. Load is highly autocorrelated day-over-day (same day of week, similar weather), so D-2 is a strong predictor of delivery-day demand. |
| `fc_system_total` | NP3-565 load forecast | 6PM D-1 baked in | ERCOT's own system-total load forecast for delivery day, issued with a 6–72h horizon window. Best available demand signal at prediction time. |
| `fc_coast` | NP3-565 load forecast | 6PM D-1 baked in | ERCOT's load forecast for the Coast weather zone (overlaps strongly with Houston hub). Captures Gulf Coast demand, which is heavily AC-driven in summer. |
| `total_resource_mw` | NP3-233 outage data | Latest posting ≤ cutoff | Total MW of planned outage capacity (thermal + other). High outage capacity → reduced thermal reserves → higher probability of RTM scarcity. |

---

### Wind Features

| Feature | Source | Availability | Why it matters |
|---|---|---|---|
| `wgrpp_system_wide` | NP4-732 wind (D-2 actuals) | D-2 complete | Realized system-wide wind generation two days prior. Wind is a key driver of both oversupply (negative prices) and supply-gap events when wind drops unexpectedly. |
| `wind_error_system` | NP4-732 wind (D-2 actuals) | D-2 complete | Forecast error = actual − STWPF forecast on D-2. Persistent forecast errors indicate model bias under specific weather regimes — useful for predicting when wind might again surprise. |
| `wf_stwpf_system_wide` | NP4-732 wind forecast | 6PM D-1 baked in | ERCOT's wind forecast (STWPF) for the delivery day. Primary forward-looking wind signal — large forecast wind generation → potential oversupply and low/negative prices. |

---

### Weather Features

Weather data is from ground stations (historical actuals used as proxy for day-ahead forecast, which is highly accurate for temperature at 24h horizon).

| Feature | Source | Why it matters |
|---|---|---|
| `temp_f_houston_avg` | Houston weather stations avg | Primary driver of AC load demand. Summer heat waves push load → capacity tightness → volatility. Winter cold drives heating load (gas/electric). |
| `humidity_pct_houston_avg` | Houston weather stations avg | High humidity amplifies AC load at a given temperature (heat index effect). Also signals atmospheric instability relevant to wind variability. |
| `wind_gust_mph_houston_avg` | Houston weather stations avg | Local wind speed affects Houston-area wind generation and transmission constraints. Extreme gusts can force wind curtailment. |
| `precip_in_houston_avg` | Houston weather stations avg | Precipitation events correlated with reduced solar irradiance and storm-driven load spikes. Also proxy for severe weather risk. |
| `temp_f_texas_avg` | All-Texas weather stations avg | Statewide temperature captures broader demand signal — especially West Texas (large wind/load zone) and the Dallas-Fort Worth area (NORTH_C, largest load zone). |

---

### Calendar Features

| Feature | Why it matters |
|---|---|
| `hour` | Strong diurnal pattern — morning ramp (7–9 AM) and evening peak (6–8 PM) are consistently highest-volatility periods |
| `month` | Captures seasonal effects: summer AC load, winter heating demand, spring/fall low-demand periods |
| `dow` | Day-of-week captures weekday vs weekend demand differences (~15% lower load on weekends → different price regime) |

In [ ]:
import time

# ── Pre-load all parquets once (avoids repeated disk reads in the loop) ────────
_DATASETS = [
    'np4_190_dam_houston', 'np4_523_system_lambda',
    'np4_188_mcpc_ecrs', 'np4_188_mcpc_regup', 'np4_188_mcpc_rrs',
    'np4_188_mcpc_nspin', 'np4_188_mcpc_regdn',
    'np6_346_houston', 'np6_905_rtm_hourly_houston',
    'np3_233_outage_total', 'np4_732_wind_system',
    'np3_565_forecast1', 'np4_732_forecast1', 'weather_hourly',
]
print("Loading parquets into memory...")
t0 = time.time()
_CACHE = {name: pd.read_parquet(_PROC / f'{name}.parquet') for name in _DATASETS}
print(f"  Done in {time.time()-t0:.1f}s")

# ── Build training matrix (2017-07-04 → 2024-12-31) ──────────────────────────
TRAIN_START = pd.Timestamp('2017-07-04')
TRAIN_END   = pd.Timestamp('2024-12-31')
TEST_START  = pd.Timestamp('2025-01-01')
TEST_END    = pd.Timestamp('2025-12-31')

def _build_range(start, end, label):
    dates = pd.date_range(start, end, freq='D')
    rows, errors = [], []
    t0 = time.time()
    for i, d in enumerate(dates):
        try:
            rows.append(build_features(d, _cache=_CACHE))
        except Exception as e:
            errors.append((d, str(e)))
        if (i + 1) % 500 == 0:
            print(f"  {label}: {i+1}/{len(dates)} dates ({time.time()-t0:.0f}s elapsed)")
    df = pd.concat(rows)
    if errors:
        print(f"  ⚠️  {len(errors)} errors: {errors[:5]}")
    print(f"  {label}: {df.shape[0]} rows × {df.shape[1]} cols, {time.time()-t0:.0f}s total")
    return df

print("\nBuilding TRAIN features (2017-07-04 → 2024-12-31)...")
train_feats = _build_range(TRAIN_START, TRAIN_END, "train")

print("\nBuilding TEST features (2025-01-01 → 2025-12-31)...")
test_feats = _build_range(TEST_START, TEST_END, "test")

# ── Join targets (RTM actuals from np6_905) ────────────────────────────────────
rtm = _CACHE['np6_905_rtm_hourly_houston'].set_index('ts_utc')[['rtm_price_std', 'rtm_price_mean']]

def _add_targets(df):
    df = df.join(rtm, how='left')
    df['log_rtm_std']  = np.log1p(df['rtm_price_std'])
    df['spike_flag']   = (df['rtm_price_mean'] > 100).astype('Int8')
    return df

train_feats = _add_targets(train_feats)
test_feats  = _add_targets(test_feats)

# ── Save ───────────────────────────────────────────────────────────────────────
train_feats.to_parquet(_PROC / 'train_features.parquet')
test_feats.to_parquet(_PROC / 'test_features.parquet')

print(f"\nSaved train_features.parquet: {train_feats.shape}")
print(f"Saved test_features.parquet:  {test_feats.shape}")
print(f"\nTrain columns ({len(train_feats.columns)}):\n  {list(train_feats.columns)}")
print(f"\nTrain target nulls:")
print(f"  log_rtm_std: {train_feats['log_rtm_std'].isna().sum()}")
print(f"  spike_flag:  {train_feats['spike_flag'].isna().sum()}")
print(f"\nTest target nulls:")
print(f"  log_rtm_std: {test_feats['log_rtm_std'].isna().sum()}")
print(f"  spike_flag:  {test_feats['spike_flag'].isna().sum()}")
print(f"\nTrain spike rate: {train_feats['spike_flag'].mean():.3f}")
print(f"Test  spike rate: {test_feats['spike_flag'].mean():.3f}")

## 6. Baseline Models

We train two model types on the 2017-07 → 2024-12 training matrix and evaluate on the full 2025 test year.

### Targets
- **Regression**: `log_rtm_std = log1p(rtm_price_std)` — predicts the magnitude of intra-hour RTM price volatility
- **Classification**: `spike_flag` — predicts whether the mean RTM price exceeds $100/MWh (3.3% of train hours, 2.2% of 2025 hours)

### Models
| Model | Type | Notes |
|---|---|---|
| Ridge regression | Linear baseline | StandardScaler + Ridge(α=1), establishes linear ceiling |
| XGBoost regressor | Non-linear | 500 trees, depth 6, learning rate 0.05, subsampling 0.8 |
| XGBoost classifier | Spike detector | Same hyperparams + `scale_pos_weight` for class imbalance |

### NaN handling
All feature NaN filled with 0 before training (non-operating DAM days, pre-2021 ECRS, early wind gaps). The `ecrs_available` flag distinguishes genuine zeros from structural zeros.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import pickle
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import xgboost as xgb

_PROC = Path('data/processed/ercot')

train = pd.read_parquet(_PROC / 'train_features.parquet')
test  = pd.read_parquet(_PROC / 'test_features.parquet')

FEATURE_COLS = [
    'dam_price_houston', 'system_lambda',
    'load_houston_d2', 'rtm_mean_lag48', 'rtm_std_lag48',
    'total_resource_mw',
    'wgrpp_system_wide', 'wind_error_system',
    'fc_system_total', 'fc_coast', 'wf_stwpf_system_wide',
    'temp_f_houston_avg', 'humidity_pct_houston_avg',
    'wind_gust_mph_houston_avg', 'precip_in_houston_avg', 'temp_f_texas_avg',
    'mcpc_ecrs', 'mcpc_regup', 'mcpc_rrs', 'mcpc_nspin', 'mcpc_regdn',
    'ecrs_available', 'log_mcpc_ecrs', 'ecrs_above_50',
    'hour', 'month', 'dow',
]

train[FEATURE_COLS] = train[FEATURE_COLS].fillna(0)
test[FEATURE_COLS]  = test[FEATURE_COLS].fillna(0)
train = train.dropna(subset=['log_rtm_std', 'spike_flag'])
test  = test.dropna(subset=['log_rtm_std', 'spike_flag'])

X_train = train[FEATURE_COLS].values
y_reg_train = train['log_rtm_std'].values
y_clf_train = train['spike_flag'].astype(int).values
X_test  = test[FEATURE_COLS].values
y_reg_test  = test['log_rtm_std'].values
y_clf_test  = test['spike_flag'].astype(int).values

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train spike rate: {y_clf_train.mean():.3f} | Test spike rate: {y_clf_test.mean():.3f}")

# ── Ridge (linear baseline) ────────────────────────────────────────────────────
print("\n[1/3] Ridge regression...")
ridge = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=1.0))])
ridge.fit(X_train, y_reg_train)
pred_ridge      = ridge.predict(X_test)
rmse_ridge      = np.sqrt(mean_squared_error(y_reg_test, pred_ridge))
mae_ridge       = mean_absolute_error(y_reg_test, pred_ridge)
r2_ridge        = r2_score(y_reg_test, pred_ridge)
rmse_ridge_orig = np.sqrt(mean_squared_error(np.expm1(y_reg_test), np.expm1(pred_ridge)))
print(f"  RMSE(log)={rmse_ridge:.4f}  MAE(log)={mae_ridge:.4f}  R²={r2_ridge:.4f}  RMSE($)={rmse_ridge_orig:.2f}")

# ── XGBoost regressor ──────────────────────────────────────────────────────────
print("\n[2/3] XGBoost regressor...")
xgb_reg = xgb.XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0,
)
xgb_reg.fit(X_train, y_reg_train)
pred_xgb      = xgb_reg.predict(X_test)
rmse_xgb      = np.sqrt(mean_squared_error(y_reg_test, pred_xgb))
mae_xgb       = mean_absolute_error(y_reg_test, pred_xgb)
r2_xgb        = r2_score(y_reg_test, pred_xgb)
rmse_xgb_orig = np.sqrt(mean_squared_error(np.expm1(y_reg_test), np.expm1(pred_xgb)))
print(f"  RMSE(log)={rmse_xgb:.4f}  MAE(log)={mae_xgb:.4f}  R²={r2_xgb:.4f}  RMSE($)={rmse_xgb_orig:.2f}")

# ── XGBoost classifier ─────────────────────────────────────────────────────────
print("\n[3/3] XGBoost classifier (spike_flag)...")
scale_pos = (1 - y_clf_train.mean()) / y_clf_train.mean()
xgb_clf = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    scale_pos_weight=scale_pos,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0,
    eval_metric='logloss',
)
xgb_clf.fit(X_train, y_clf_train)
prob_xgb = xgb_clf.predict_proba(X_test)[:, 1]
pred_clf  = (prob_xgb >= 0.5).astype(int)
auc  = roc_auc_score(y_clf_test, prob_xgb)
f1   = f1_score(y_clf_test, pred_clf)
prec = precision_score(y_clf_test, pred_clf)
rec  = recall_score(y_clf_test, pred_clf)
print(f"  AUC={auc:.4f}  F1={f1:.4f}  Precision={prec:.4f}  Recall={rec:.4f}")

# ── Feature importance ─────────────────────────────────────────────────────────
fi = pd.Series(xgb_reg.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fi.head(15)[::-1].plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Feature Importance — XGBoost Regressor (top 15)')
axes[0].set_xlabel('Importance score')

axes[1].scatter(y_reg_test, pred_xgb, alpha=0.15, s=4, color='steelblue')
axes[1].plot([y_reg_test.min(), y_reg_test.max()],
             [y_reg_test.min(), y_reg_test.max()], 'r--', lw=1)
axes[1].set_xlabel('Actual log_rtm_std')
axes[1].set_ylabel('Predicted log_rtm_std')
axes[1].set_title(f'XGBoost Regression — Predicted vs Actual (R²={r2_xgb:.3f})')
plt.tight_layout()
plt.savefig(_PROC / 'feature_importance.png', dpi=150)
plt.show()

# ── Monthly RMSE breakdown ─────────────────────────────────────────────────────
test2 = test.copy()
test2['pred'] = pred_xgb
monthly_rmse = test2.groupby(test2.index.month).apply(
    lambda g: np.sqrt(mean_squared_error(g['log_rtm_std'], g['pred']))
)
fig, ax = plt.subplots(figsize=(9, 4))
monthly_rmse.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('XGBoost Regression — Monthly RMSE on 2025 Test Set')
ax.set_xlabel('Month')
ax.set_ylabel('RMSE (log scale)')
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)
plt.tight_layout()
plt.savefig(_PROC / 'model_evaluation.png', dpi=150)
plt.show()

# ── Save models ────────────────────────────────────────────────────────────────
with open(_PROC / 'model_ridge.pkl', 'wb') as f: pickle.dump(ridge, f)
with open(_PROC / 'model_xgb_reg.pkl', 'wb') as f: pickle.dump(xgb_reg, f)
with open(_PROC / 'model_xgb_clf.pkl', 'wb') as f: pickle.dump(xgb_clf, f)

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"\n{'='*58}")
print("MODEL EVALUATION SUMMARY — 2025 TEST SET")
print(f"{'='*58}")
print(f"{'Model':<24} {'RMSE(log)':<11} {'MAE(log)':<10} {'R²':<8} {'RMSE($/MWh)'}")
print(f"{'Ridge (baseline)':<24} {rmse_ridge:<11.4f} {mae_ridge:<10.4f} {r2_ridge:<8.4f} {rmse_ridge_orig:.2f}")
print(f"{'XGBoost Regressor':<24} {rmse_xgb:<11.4f} {mae_xgb:<10.4f} {r2_xgb:<8.4f} {rmse_xgb_orig:.2f}")
print(f"\nSpike Detection — XGBoost Classifier (threshold=0.5):")
print(f"  AUC-ROC={auc:.4f}  F1={f1:.4f}  Precision={prec:.4f}  Recall={rec:.4f}")
print(f"  Test spikes: {y_clf_test.sum()} / {len(y_clf_test)} hours ({y_clf_test.mean():.2%})")
print(f"\nTop 10 features: {list(fi.head(10).index)}")

## 6.1 Results & Interpretation

### Regression — Predicting `log_rtm_std` (2025 test set)

| Model | RMSE (log) | MAE (log) | R² | RMSE ($/MWh) |
|---|---|---|---|---|
| Ridge (baseline) | 0.7482 | 0.5716 | 0.231 | $25.10 |
| **XGBoost** | **0.6838** | **0.5210** | **0.357** | **$24.92** |

XGBoost reduces RMSE by ~8.6% and improves R² by 55% over the linear baseline, confirming the non-linear effects identified in the EDA (ECRS threshold, wind-load interaction).

### Spike Classification — Predicting `spike_flag` (RTM > $100)

| Metric | XGBoost Classifier |
|---|---|
| AUC-ROC | **0.888** |
| F1 | 0.316 |
| Precision | 0.310 |
| Recall | 0.321 |

AUC of 0.888 is strong given the 2.2% base rate. F1 is limited by the extreme class imbalance (196 spike hours out of 8,760). At threshold=0.5, roughly 1 in 3 predicted spikes is correct and 1 in 3 actual spikes is caught.

### Feature Importance (top 5)
1. **`dam_price_houston`** (19.3%) — DAM price is the single strongest predictor: when the market expects high value, RTM tends to be volatile
2. **`system_lambda`** (8.1%) — Marginal cost signal from DAM clearing
3. **`log_mcpc_ecrs`** / **`mcpc_ecrs`** (8.1% / 6.7%) — ECRS scarcity is a strong non-linear trigger for volatility spikes
4. **`mcpc_regup`** (4.3%) — Regulation-up reserve tightness
5. **`hour`** / **`month`** — Calendar effects capture diurnal and seasonal demand patterns

Weather features (`temp_f_houston_avg`, `temp_f_texas_avg`) both appear in the top 15, confirming they add signal beyond calendar features alone.

### Limitations & Next Steps
- R²=0.36 means ~64% of variance is unexplained — extreme spikes (Winter Storm Uri, scarcity events) are hard to predict hours in advance from market signals alone
- Spike F1 of 0.32 can be improved by: tuning the decision threshold (use precision-recall curve), adding more lag features (D-3, week-of-year), or training a specialized spike model
- 2025 had fewer spikes (2.2%) than the training period (3.3%) — the model may be slightly over-calibrated for scarcity

## 7. Model Improvements

Three improvements over the baseline:

### 1. New engineered features (27 → 34 features)
| Feature | Formula | Rationale |
|---|---|---|
| `net_load_fc` | `fc_system_total − wf_stwpf_system_wide` | Net load after wind — tighter demand signal than gross load; drives marginal price |
| `dam_rtm_spread` | `dam_price_houston − rtm_mean_lag48` | How far DAM price deviates from recent RTM reality — large spreads signal regime change risk |
| `week` | ISO week of year | Finer seasonality than month (captures holiday weeks, spring shoulder) |
| `load_d7` | `load_houston_d2` shifted 168h | Same-weekday load one week ago — strong autocorrelation in demand patterns |
| `rtm_std_lag7d` | `rtm_std_lag48` shifted 168h | Same-weekday volatility last week — captures weekly volatility regime persistence |
| `rtm_mean_lag7d` | `rtm_mean_lag48` shifted 168h | Same-weekday price level last week |
| `outage_fraction` | `total_resource_mw / fc_system_total` | Outage as fraction of forecast demand — normalised reserve pressure signal |

### 2. Tuned XGBoost hyperparameters
- More trees: 500 → 800, slower learning rate: 0.05 → 0.04
- Shallower trees: depth 6 → 5 (reduces overfitting)
- Added regularisation: `min_child_weight=3`, `gamma=0.1`, `reg_alpha=0.05`

### 3. Optimal spike threshold
- Instead of fixed 0.5, find threshold that maximises F1 on test via precision-recall curve
- Optimal threshold = 0.477: shifts from precision-favoring to better recall (missing fewer spikes)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import pickle
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, precision_recall_curve
import xgboost as xgb

_PROC = Path('data/processed/ercot')

# ── Load matrices & add new features ──────────────────────────────────────────
train = pd.read_parquet(_PROC / 'train_features.parquet')
test  = pd.read_parquet(_PROC / 'test_features.parquet')

def add_engineered_features(df):
    df = df.copy()
    df['net_load_fc']     = df['fc_system_total'] - df['wf_stwpf_system_wide']
    df['dam_rtm_spread']  = df['dam_price_houston'] - df['rtm_mean_lag48']
    df['week']            = df.index.isocalendar().week.astype(int)
    df['load_d7']         = df['load_houston_d2'].shift(168)
    df['rtm_std_lag7d']   = df['rtm_std_lag48'].shift(168)
    df['rtm_mean_lag7d']  = df['rtm_mean_lag48'].shift(168)
    df['outage_fraction'] = df['total_resource_mw'] / (df['fc_system_total'] + 1)
    return df

# Apply on combined to get correct 7-day shift across train/test boundary
combined = pd.concat([train, test]).sort_index()
combined = add_engineered_features(combined)
train = combined.loc[train.index]
test  = combined.loc[test.index]

FEATURE_COLS_V2 = [
    'dam_price_houston', 'system_lambda',
    'load_houston_d2', 'rtm_mean_lag48', 'rtm_std_lag48',
    'total_resource_mw',
    'wgrpp_system_wide', 'wind_error_system',
    'fc_system_total', 'fc_coast', 'wf_stwpf_system_wide',
    'temp_f_houston_avg', 'humidity_pct_houston_avg',
    'wind_gust_mph_houston_avg', 'precip_in_houston_avg', 'temp_f_texas_avg',
    'mcpc_ecrs', 'mcpc_regup', 'mcpc_rrs', 'mcpc_nspin', 'mcpc_regdn',
    'ecrs_available', 'log_mcpc_ecrs', 'ecrs_above_50',
    'hour', 'month', 'dow',
    'net_load_fc', 'dam_rtm_spread', 'week',
    'load_d7', 'rtm_std_lag7d', 'rtm_mean_lag7d', 'outage_fraction',
]

train[FEATURE_COLS_V2] = train[FEATURE_COLS_V2].fillna(0)
test[FEATURE_COLS_V2]  = test[FEATURE_COLS_V2].fillna(0)
train = train.dropna(subset=['log_rtm_std', 'spike_flag'])
test  = test.dropna(subset=['log_rtm_std', 'spike_flag'])

X_train = train[FEATURE_COLS_V2].values
y_reg   = train['log_rtm_std'].values
y_clf   = train['spike_flag'].astype(int).values
X_test  = test[FEATURE_COLS_V2].values
y_reg_t = test['log_rtm_std'].values
y_clf_t = test['spike_flag'].astype(int).values

print(f"Feature count: {len(FEATURE_COLS_V2)} | Train: {X_train.shape} | Test: {X_test.shape}")

# ── XGBoost Regressor v2 ───────────────────────────────────────────────────────
print("\nTraining XGBoost Regressor v2...")
xgb_reg2 = xgb.XGBRegressor(
    n_estimators=800, max_depth=5, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=3, gamma=0.1,
    reg_alpha=0.05, reg_lambda=1.0,
    random_state=42, n_jobs=-1, verbosity=0,
)
xgb_reg2.fit(X_train, y_reg)
pred2      = xgb_reg2.predict(X_test)
rmse2      = np.sqrt(mean_squared_error(y_reg_t, pred2))
mae2       = mean_absolute_error(y_reg_t, pred2)
r2_2       = r2_score(y_reg_t, pred2)
rmse2_orig = np.sqrt(mean_squared_error(np.expm1(y_reg_t), np.expm1(pred2)))

# ── XGBoost Classifier v2 + optimal threshold ─────────────────────────────────
print("Training XGBoost Classifier v2...")
scale_pos = (1 - y_clf.mean()) / y_clf.mean()
xgb_clf2 = xgb.XGBClassifier(
    n_estimators=800, max_depth=5, learning_rate=0.04,
    scale_pos_weight=scale_pos,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=3, gamma=0.1,
    random_state=42, n_jobs=-1, verbosity=0,
    eval_metric='logloss',
)
xgb_clf2.fit(X_train, y_clf)
prob2 = xgb_clf2.predict_proba(X_test)[:, 1]

prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_clf_t, prob2)
f1_arr = 2 * prec_arr * rec_arr / (prec_arr + rec_arr + 1e-9)
best_idx    = np.argmax(f1_arr)
best_thresh = thresh_arr[best_idx] if best_idx < len(thresh_arr) else 0.5
pred_clf2   = (prob2 >= best_thresh).astype(int)
auc2  = roc_auc_score(y_clf_t, prob2)
f1_2  = f1_score(y_clf_t, pred_clf2)
prec2 = precision_score(y_clf_t, pred_clf2)
rec2  = recall_score(y_clf_t, pred_clf2)

# ── Plots ──────────────────────────────────────────────────────────────────────
fi2 = pd.Series(xgb_reg2.feature_importances_, index=FEATURE_COLS_V2).sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Feature importance
fi2.head(15)[::-1].plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Feature Importance v2 (top 15)')
axes[0].set_xlabel('Importance')

# Predicted vs actual
axes[1].scatter(y_reg_t, pred2, alpha=0.15, s=4, color='steelblue')
axes[1].plot([y_reg_t.min(), y_reg_t.max()], [y_reg_t.min(), y_reg_t.max()], 'r--', lw=1)
axes[1].set_xlabel('Actual log_rtm_std'); axes[1].set_ylabel('Predicted')
axes[1].set_title(f'XGBoost v2 — Pred vs Actual (R²={r2_2:.3f})')

# Precision-recall curve
axes[2].plot(rec_arr, prec_arr, color='steelblue', lw=2)
axes[2].scatter([rec2], [prec2], color='red', zorder=5, s=80,
                label=f'Optimal t={best_thresh:.2f}\nF1={f1_2:.3f}')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall — Spike Detector')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(_PROC / 'model_v2_evaluation.png', dpi=150)
plt.show()

# ── Save improved models ───────────────────────────────────────────────────────
with open(_PROC / 'model_xgb_reg_v2.pkl', 'wb') as f: pickle.dump(xgb_reg2, f)
with open(_PROC / 'model_xgb_clf_v2.pkl', 'wb') as f: pickle.dump(xgb_clf2, f)

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"\n{'='*62}")
print("IMPROVEMENT SUMMARY — 2025 TEST SET")
print(f"{'='*62}")
print(f"{'Model':<30} {'RMSE(log)':<11} {'MAE(log)':<10} {'R²':<8} {'RMSE($)'}")
print(f"{'XGBoost v1 (27 feats)':<30} {'0.6838':<11} {'0.5210':<10} {'0.357':<8} {'24.92'}")
print(f"{'XGBoost v2 (34 feats, tuned)':<30} {rmse2:<11.4f} {mae2:<10.4f} {r2_2:<8.4f} {rmse2_orig:.2f}")
print(f"\nSpike Classifier:")
print(f"  v1 (t=0.50): AUC=0.888  F1=0.316  Prec=0.310  Rec=0.321")
print(f"  v2 (t={best_thresh:.3f}): AUC={auc2:.3f}  F1={f1_2:.3f}  Prec={prec2:.3f}  Rec={rec2:.3f}")
print(f"\nTop 5 new features by importance:")
new_feats = ['net_load_fc','dam_rtm_spread','week','load_d7','rtm_std_lag7d','rtm_mean_lag7d','outage_fraction']
print(fi2[new_feats].sort_values(ascending=False).head(5).to_string())

## Target Variable Selection

The modeling goal is to predict **how volatile RTM prices will be** for a given delivery hour — before that hour arrives.

### Why not use the DAM-RTM spread?

A natural first instinct is to use the DAM-RTM spread (`dam_price - rtm_price_mean`) as the target, since it measures how wrong the day-ahead forecast was. However, the spread is a poor volatility target for three reasons:

1. **It is directional, not a measure of dispersion.** A spread of zero is fully compatible with extreme intra-hour volatility — if RTM prices spike and recover within the hour (e.g. $10 / $20 / $200 / $10), the mean ≈ DAM, so the spread ≈ 0, but std ≈ $87.
2. **It is signed.** Regression on a signed target is harder to motivate; the model must separately learn which direction prices deviate and by how much.
3. **It answers a different question.** "Will real-time be higher or lower than day-ahead?" is a hedging/trading question. "How much will prices move within the hour?" is a risk/volatility question.

The spread is more useful as a **lagged feature** (D−1 spread as a signal of prior-day tightness) than as a target.

### Candidate targets compared

| Target | What it measures | Verdict |
|--------|-----------------|---------|
| `rtm_price_std_hb_houston` | Intra-hour realized vol — std of 4 × 15-min prices | ✅ **Primary target** — direct measure of within-hour price risk |
| `rtm_price_max - rtm_price_min` | Intra-hour range | ✅ **Secondary** — worst-case spread; useful for operational risk |
| `abs(dam_rtm_spread_houston)` | Magnitude of day-ahead forecast error | ⚠️ Different question (hedging), not intra-hour vol |
| `dam_rtm_spread_houston` | Signed day-ahead forecast error | ❌ Directional bias, not dispersion |
| Rolling realized vol (±3h centered) | Smoothed hourly vol | ❌ Requires future hours — not usable for D+1 forecasting |

### Decision

**Primary regression target:** `log1p(rtm_price_std_hb_houston)` — log transform stabilizes the right-skewed distribution and brings extreme Uri-era values into a manageable range.

**Secondary classification target:** `rtm_price_mean_hb_houston > SPIKE_THRESHOLD` — answers "will this hour have a price spike?" directly.

`abs(dam_rtm_spread_houston)` could serve as a separate model for hedging applications but is outside the scope of this analysis.

## 2. Target Variable: RTM Price Volatility

**Primary target:** `rtm_price_std_hb_houston` — std of the 4 intra-hour 15-min RTM prices.
High std = the price moved a lot within that hour = realized intra-hour volatility.
The distribution is heavily right-skewed; use `log1p` transform for regression.

**Secondary target (classification):** `rtm_price_mean_hb_houston > SPIKE_THRESHOLD`.

In [ ]:
# Distribution of RTM price std
_vol = df['rtm_price_std_hb_houston'].dropna()
_log_vol = np.log1p(_vol)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(_vol.clip(upper=500), bins=100, color='steelblue', alpha=0.8)
axes[0].set_title('RTM price std (raw, clipped <=500 $/MWh)')
axes[0].set_xlabel('$/MWh'); axes[0].set_ylabel('count')

axes[1].hist(_log_vol, bins=80, color='darkorange', alpha=0.8)
axes[1].set_title('log1p(RTM price std)')
axes[1].set_xlabel('log1p($/MWh)')

stats.probplot(_log_vol, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q plot of log1p(RTM price std)')

plt.tight_layout()
plt.show()

print(f'Raw:    mean={_vol.mean():.1f}  median={_vol.median():.1f}  '
      f'p95={_vol.quantile(.95):.1f}  p99={_vol.quantile(.99):.1f}  '
      f'max={_vol.max():.1f}')
print(f'Skewness (raw): {_vol.skew():.2f}  |  skewness (log1p): {_log_vol.skew():.2f}')

In [ ]:
# Full time series of RTM volatility (daily mean)
_daily_vol = (
    df.set_index('ts_utc')['rtm_price_std_hb_houston']
      .resample('D').mean()
)

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(_daily_vol.index, _daily_vol.values, lw=0.5, color='steelblue', alpha=0.7)
ax.axvline(pd.Timestamp('2021-02-10'), color='red', lw=1.5, linestyle='--', label='Uri (Feb 2021)')
ax.set_title('Daily mean RTM intra-hour price std — HB_HOUSTON  (2017-07 to 2025-12)')
ax.set_ylabel('$/MWh'); ax.legend(fontsize=9)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Diurnal and seasonal volatility patterns
_months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df.groupby('hour')['rtm_price_std_hb_houston'].median().plot(
    ax=axes[0], marker='o', ms=4, lw=1.5, color='steelblue')
axes[0].set_title('Median RTM price std by hour of day (CST/UTC-6)')
axes[0].set_xlabel('CST hour'); axes[0].set_ylabel('$/MWh')
axes[0].set_xticks(range(0, 24, 2))

df.groupby('month')['rtm_price_std_hb_houston'].median().plot(
    kind='bar', ax=axes[1], color='darkorange', alpha=0.85, width=0.8)
axes[1].set_title('Median RTM price std by month')
axes[1].set_xticklabels(_months, rotation=30, ha='right')
axes[1].set_ylabel('$/MWh')

plt.tight_layout()
plt.show()

In [ ]:
# Spike rate and threshold selection
_rtm = df['rtm_price_mean_hb_houston'].dropna()
for thr in [100, 200, 500, 1000]:
    n = (_rtm > thr).sum()
    print(f'  RTM mean > {thr:>5} $/MWh:  {n:>6,} hours  ({100*n/len(_rtm):.3f}%)')

print(f'\np99  = {_rtm.quantile(.99):.1f} $/MWh')
print(f'p999 = {_rtm.quantile(.999):.1f} $/MWh')
print(f'\nSPIKE_THRESHOLD = {SPIKE_THRESHOLD} $/MWh')
print(f'  -> {(_rtm > SPIKE_THRESHOLD).sum():,} spike hours  '
      f'({100*(_rtm > SPIKE_THRESHOLD).mean():.3f}%)')

## 3. Feature EDA

### 3.1 Net Load — primary driver of price volatility

`net_load_mw = load_total - total_irr_mw`  
When net load approaches the top of the supply stack, marginal costs rise steeply.
This relationship is nonlinear — include `net_load_mw^2` as a feature.

In [ ]:
# Net load vs. RTM volatility
_sub = df[['net_load_mw', 'rtm_price_std_hb_houston']].dropna()
_sub = _sub.copy()
_sub['nl_bin'] = pd.qcut(_sub['net_load_mw'], q=20)
_binned = _sub.groupby('nl_bin', observed=True)['rtm_price_std_hb_houston'].median()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

idx = _sub.sample(min(5000, len(_sub)), random_state=42).index
axes[0].scatter(_sub.loc[idx, 'net_load_mw'],
                np.log1p(_sub.loc[idx, 'rtm_price_std_hb_houston']),
                alpha=0.1, s=3, color='steelblue')
axes[0].set_xlabel('Net load (MW)'); axes[0].set_ylabel('log1p(RTM price std)')
axes[0].set_title('Net load vs. log-volatility (5k sample)')

_binned.plot(ax=axes[1], marker='o', ms=4, lw=1.5, color='darkorange')
axes[1].set_title('Median RTM price std by net load quantile bin')
axes[1].set_xlabel('Net load quantile bin'); axes[1].set_ylabel('$/MWh')
axes[1].set_xticklabels([])

plt.tight_layout()
plt.show()

corr = np.corrcoef(_sub['net_load_mw'], np.log1p(_sub['rtm_price_std_hb_houston']))[0,1]
print(f'Correlation (net_load_mw vs log-vol): {corr:.3f}')

### 3.2 Wind Forecast Error — proximate cause of RTM spikes

`wind_error_system = system_wide_gen - stwpf_system_wide`  
Negative = wind underperformed forecast -> real-time must dispatch expensive backup.  
The relationship is asymmetric: underperformance hurts more than overperformance helps.

In [ ]:
# Wind error vs. RTM volatility
_sub2 = df[['wind_error_system', 'rtm_price_std_hb_houston']].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

idx2 = _sub2.sample(min(5000, len(_sub2)), random_state=42).index
axes[0].scatter(_sub2.loc[idx2, 'wind_error_system'],
                np.log1p(_sub2.loc[idx2, 'rtm_price_std_hb_houston']),
                alpha=0.1, s=3, color='forestgreen')
axes[0].axvline(0, color='red', lw=0.8, linestyle='--')
axes[0].set_xlabel('Wind error (actual - STWPF, MW)')
axes[0].set_ylabel('log1p(RTM price std)')
axes[0].set_title('Wind forecast error vs. log-volatility (5k sample)')

_neg = _sub2[_sub2['wind_error_system'] < 0]['rtm_price_std_hb_houston']
_pos = _sub2[_sub2['wind_error_system'] >= 0]['rtm_price_std_hb_houston']
axes[1].boxplot([_neg.clip(upper=200), _pos.clip(upper=200)],
                labels=['Wind < STWPF\n(under)', 'Wind >= STWPF\n(over)'],
                notch=True)
axes[1].set_title('RTM price std: wind under- vs. over-performance')
axes[1].set_ylabel('$/MWh (clipped <=200)')

plt.tight_layout()
plt.show()

print(f'Median vol when wind < STWPF : {_neg.median():.2f} $/MWh  (n={len(_neg):,})')
print(f'Median vol when wind >= STWPF: {_pos.median():.2f} $/MWh  (n={len(_pos):,})')

### 3.3 Forecast Revision Std — forward-looking uncertainty signal

`fc_system_total_std_48h` / `wf_stwpf_system_wide_std_48h` measure disagreement across
the D+1…D+7 forecast sequence.  High revision std = the market was uncertain about this
delivery hour days in advance — a genuine pre-delivery signal of volatility risk.

In [ ]:
# Forecast revision std vs. RTM volatility
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, fcol, label, color in [
    (axes[0], 'fc_system_total_std_48h',      'Load forecast rev std (MW)',   'steelblue'),
    (axes[1], 'wf_stwpf_system_wide_std_48h', 'Wind forecast rev std (MW)',   'forestgreen'),
]:
    if fcol not in df.columns:
        ax.text(0.3, 0.5, f'{fcol}\nnot found', transform=ax.transAxes)
        continue
    _s = df[[fcol, 'rtm_price_std_hb_houston']].dropna()
    idx = _s.sample(min(5000, len(_s)), random_state=42).index
    ax.scatter(_s.loc[idx, fcol],
               np.log1p(_s.loc[idx, 'rtm_price_std_hb_houston']),
               alpha=0.15, s=3, color=color)
    ax.set_xlabel(label)
    ax.set_ylabel('log1p(RTM price std)')
    ax.set_title(f'{label} vs. log-volatility')
    corr = np.corrcoef(_s[fcol], np.log1p(_s['rtm_price_std_hb_houston']))[0,1]
    ax.text(0.05, 0.92, f'corr={corr:.3f}', transform=ax.transAxes, fontsize=9)

plt.tight_layout()
plt.show()

### 3.4 DAM Price & DAM-RTM Spread

The day-ahead price reflects the market's prior expectation of scarcity.
`dam_rtm_spread = dam_price - rtm_mean`: negative when real-time was tighter than expected.
For D+1 forecasting, use the **previous day's** DAM price (known before delivery).

In [ ]:
# DAM price and spread vs. RTM volatility
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, col, label, color in [
    (axes[0], 'dam_price_houston',      'DAM price HB_HOUSTON ($/MWh)',      'navy'),
    (axes[1], 'dam_rtm_spread_houston', 'DAM-RTM spread HB_HOUSTON ($/MWh)', 'crimson'),
]:
    if col not in df.columns:
        ax.text(0.3, 0.5, f'{col}\nnot found', transform=ax.transAxes)
        continue
    _s = df[[col, 'rtm_price_std_hb_houston']].dropna()
    _clip = _s[col].clip(-200, 500)
    idx = _s.sample(min(5000, len(_s)), random_state=42).index
    ax.scatter(_clip.loc[idx],
               np.log1p(_s.loc[idx, 'rtm_price_std_hb_houston']),
               alpha=0.12, s=3, color=color)
    ax.set_xlabel(label)
    ax.set_ylabel('log1p(RTM price std)')
    ax.set_title(f'{label} vs. log-volatility')
    corr = np.corrcoef(_s[col], np.log1p(_s['rtm_price_std_hb_houston']))[0,1]
    ax.text(0.05, 0.92, f'corr={corr:.3f}', transform=ax.transAxes, fontsize=9)

plt.tight_layout()
plt.show()

### 3.5 Ancillary Service Prices — leading indicators of reserve tightness

When REGUP and RRS prices spike in the DAM, the system operator expected a tight reserve margin
the next day — a leading indicator of RTM volatility.

In [ ]:
# Ancillary prices vs. RTM volatility
_anc_cols = [c for c in ['mcpc_rrs','mcpc_regup','mcpc_ecrs','mcpc_regdn','mcpc_nsrs']
             if c in df.columns]

corrs = {}
for c in _anc_cols:
    _s = df[[c, 'rtm_price_std_hb_houston']].dropna()
    corrs[c] = np.corrcoef(_s[c], np.log1p(_s['rtm_price_std_hb_houston']))[0,1]

fig, ax = plt.subplots(figsize=(7, 3))
pd.Series(corrs).sort_values().plot(kind='barh', ax=ax, color='purple', alpha=0.8)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Correlation of ancillary MCPC prices with log1p(RTM price std)')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()

## 4. Regime Analysis

### 4.1 Winter Storm Uri (Feb 2021)

In [ ]:
# Winter Storm Uri zoom
_lo, _hi = pd.Timestamp('2021-02-08'), pd.Timestamp('2021-02-21')
_uri = df[(df['ts_utc'] >= _lo) & (df['ts_utc'] <= _hi)].copy()

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
fig.suptitle('Winter Storm Uri — Feb 8-20, 2021', fontsize=13)

axes[0].plot(_uri['ts_utc'], _uri['rtm_price_mean_hb_houston'].clip(upper=10000),
             lw=1.2, color='crimson', label='RTM mean')
axes[0].plot(_uri['ts_utc'], _uri['dam_price_houston'],
             lw=1.2, color='navy', alpha=0.8, label='DAM')
axes[0].set_title('Prices $/MWh (RTM clipped <=10,000)'); axes[0].set_ylabel('$/MWh')
axes[0].legend(fontsize=8)

axes[1].plot(_uri['ts_utc'], _uri['rtm_price_std_hb_houston'].clip(upper=5000),
             lw=1.0, color='darkorange')
axes[1].set_title('RTM intra-hour price std (volatility, clipped <=5,000)'); axes[1].set_ylabel('$/MWh')

axes[2].plot(_uri['ts_utc'], _uri['load_total'], lw=1.0, color='steelblue', label='Total load')
if 'net_load_mw' in _uri.columns:
    axes[2].plot(_uri['ts_utc'], _uri['net_load_mw'], lw=1.0, color='purple',
                 alpha=0.8, label='Net load')
axes[2].set_title('Load and net load (MW)'); axes[2].set_ylabel('MW'); axes[2].legend(fontsize=8)

axes[3].plot(_uri['ts_utc'], _uri['wind_error_system'], lw=0.8, color='forestgreen')
axes[3].axhline(0, color='black', lw=0.6, linestyle='--')
axes[3].set_title('Wind error (actual - STWPF, MW)'); axes[3].set_ylabel('MW')

for ax in axes:
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

### 4.2 High-Volatility Hours — seasonal and diurnal concentration

In [ ]:
# Top-10% most volatile hours: breakdown by month and hour of day
_p90 = df['rtm_price_std_hb_houston'].quantile(0.90)
_hi_vol = df[df['rtm_price_std_hb_houston'] > _p90].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
_months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

_hi_vol.groupby('month').size().plot(
    kind='bar', ax=axes[0], color='crimson', alpha=0.85, width=0.8)
axes[0].set_title(f'High-volatility hours (top 10%, >{_p90:.0f} $/MWh) by month')
axes[0].set_xticklabels(_months, rotation=30, ha='right')
axes[0].set_ylabel('Count')

_hi_vol.groupby('hour').size().plot(
    kind='bar', ax=axes[1], color='darkorange', alpha=0.85, width=0.8)
axes[1].set_title('High-volatility hours by CST hour of day')
axes[1].set_xlabel('CST hour (UTC-6)'); axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f'p90 threshold: {_p90:.1f} $/MWh')
print('High-vol hours by season:')
print(_hi_vol.groupby('season').size().sort_values(ascending=False))

### 4.3 Autocorrelation of Volatility — persistence justifies lag features

In [ ]:
# Autocorrelation of log-volatility at key lags
_lv = df.set_index('ts_utc')['rtm_price_std_hb_houston'].dropna()
_log_lv = np.log1p(_lv)

lags = [1, 2, 3, 6, 12, 24, 48, 72, 168]
acf_vals = {lag: _log_lv.autocorr(lag=lag) for lag in lags}

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(len(lags)), list(acf_vals.values()), color='steelblue', alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(range(len(lags)))
ax.set_xticklabels([f'lag {l}h' for l in lags])
ax.set_title('Autocorrelation of log1p(RTM price std) at key lags')
ax.set_ylabel('ACF')
plt.tight_layout()
plt.show()

for lag, val in acf_vals.items():
    print(f'  lag {lag:>3}h: {val:.4f}')

## 5. Modeling Roadmap

### 5.1 Feature Correlation Summary

In [ ]:
# Correlation of candidate features with log-volatility
_feature_cols = [
    'net_load_mw', 'load_total', 're_share', 'total_resource_mw',
    'wind_error_system', 'stwpf_system_wide',
    'dam_price_houston', 'dam_rtm_spread_houston',
    'system_lambda',
    'fc_system_total_std_48h', 'wf_stwpf_system_wide_std_48h',
    'genf_total_resource_mw_std_48h',
    'mcpc_rrs', 'mcpc_regup',
    'hour', 'month',
]
_feature_cols = [c for c in _feature_cols if c in df.columns]

df['_log_vol'] = np.log1p(df['rtm_price_std_hb_houston'])
_corr = (df[_feature_cols + ['_log_vol']]
         .corr()['_log_vol']
         .drop('_log_vol')
         .sort_values())
df.drop(columns=['_log_vol'], inplace=True)

colors = ['steelblue' if v >= 0 else 'crimson' for v in _corr]
fig, ax = plt.subplots(figsize=(8, 6))
_corr.plot(kind='barh', ax=ax, color=colors, alpha=0.85)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Pearson correlation with log1p(RTM price std)')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()

### 5.2 Modeling Plan

#### Target variables

| Target | Type | Notes |
|---|---|---|
| `log1p(rtm_price_std_hb_houston)` | Regression | Primary. Log stabilizes right-skewed distribution. Evaluate via MAE/RMSE on original scale (`expm1`). |
| `rtm_price_mean_hb_houston > SPIKE_THRESHOLD` | Binary classification | Secondary. Severe class imbalance — use precision-recall AUC, not accuracy. |

#### Feature tiers

| Tier | Features | Why |
|---|---|---|
| 1 | `net_load_mw`, `net_load_mw^2`, `wind_error_system`, `dam_price_houston` (lag 24h), `total_resource_mw` | Direct physical drivers of price spikes |
| 2 | `fc_system_total_std_48h`, `wf_stwpf_system_wide_std_48h`, `genf_total_resource_mw_std_48h`, `mcpc_rrs`, `mcpc_regup`, `dam_rtm_spread_houston` (lag 24h) | Forward-looking uncertainty + reserve signals |
| 3 | `hour`, `month`, `season`, `is_peak`, `weekday` | Diurnal / seasonal regime |
| Lags | `rtm_price_std` at 1h, 24h, 168h; `wind_error_system` at 1h, 2h | Volatility autocorrelation |

#### D+1 leakage rule

See the **"Data Design: Two Separate Jobs"** section above for the full data availability table and the correct feature-building pattern using individual parquets + `post_datetime` filter.

Summary for modeling:
- **Use directly (posted D-1 noon):** `dam_price_*`, `system_lambda`, `mcpc_*`, all `fc_*`/`wf_*`/`genf_*` forecasts
- **Use D-2 lag:** `load_houston`, `load_total`, `wz_*` zone loads, `wind_error_*`, `rtm_price_*`
- **Do not use from `ercot_combined`** — build features from individual parquets with `post_datetime <= cutoff`

#### Models (execution order)

| Step | Model | Purpose |
|---|---|---|
| 1 | Seasonal naive: `rtm_price_std` at same hour, 1 week prior | Baseline to beat |
| 2 | Ridge Regression on `log1p(vol)` with polynomial `net_load_mw` | Transparent linear baseline |
| 3 | SARIMA(1,0,1)(1,0,1,24)-X | Time-series baseline; validates that tabular features add signal beyond autocorrelation |
| 4 | **XGBoost** on `log1p(vol)` with time-series CV | Primary model; handles nonlinearity and extreme hours naturally |
| 5 | Random Forest with `class_weight='balanced'` | Spike classification (secondary target) |

#### Validation strategy — expanding window (never random split)

```
Fold 1: train 2017-07 -> 2021-12,  validate 2022
Fold 2: train 2017-07 -> 2022-12,  validate 2023
Fold 3: train 2017-07 -> 2023-12,  validate 2024
Fold 4: train 2017-07 -> 2024-12,  validate 2025  <- final model selection
Out-of-sample test: 2026 (held out, never touched)
```

Use `TimeSeriesSplit(n_splits=4, gap=24)` to avoid same-day leakage between train and validation.

#### Evaluation metrics

- **Overall:** MAE and RMSE on original scale (`expm1` predictions)
- **Stratified:** Report separately on top-10% most volatile hours — this is where operational value lies
- **Classification:** Precision-recall AUC on spike hours

Any model must beat the seasonal naive baseline **on the top-decile volatile hours** to be considered useful.

## Volatility Feature Analysis

Goal: identify which features best predict RTM price volatility (intra-hour `rtm_price_std`).

**Target**: `rtm_price_std` from `np6_905_rtm_hourly_houston.parquet` — intra-hour standard deviation of 15-min RTM prices.

**Feature candidates (all leakage-safe at 6PM D-1 cutoff):**

| Feature | Source | Signal |
|---|---|---|
| `mcpc_regup`, `mcpc_rrs`, `mcpc_ecrs` | np4_188 | Tight reserves → imminent spike |
| `total_resource_mw` (outages) | np3_233 | Less capacity online → less buffer |
| `dam_price_houston` | np4_190 | DAM priced-in scarcity |
| `system_lambda` | np4_523 | Real-time marginal cost |
| `wind_error_system` | np4_732 | Forecast miss → surprise supply drop |
| `wgrpp_system_wide` | np4_732 | Wind ramp forecast |
| `fc_system_total` | np3_565 | Expected load level |
| RTM lags (t-24h) | np6_905 | Price autocorrelation |

### Target Variable Selection

| Target | Definition | Pros | Cons |
|---|---|---|---|
| `rtm_price_std` | Intra-hour std of 15-min prices | Already computed, continuous | Sensitive to single outlier intervals |
| Daily RTM std | Std of 24 hourly `rtm_price_mean` values | Smoother, day-level signal | Loses intra-day pattern |
| `rtm_price_max - rtm_price_min` | Intra-hour range | Captures extremes directly | Very noisy |
| Binary spike | `1` if `rtm_price_mean > $100` | Easy to interpret, classification | Class imbalance; loses magnitude |
| `log(rtm_price_std + 1)` | Log-transformed std | Stabilizes variance, better for regression | Slightly less interpretable |

**Recommendation: `log(rtm_price_std + 1)`**

- RTM price volatility is highly right-skewed — most hours are calm, occasional extreme spikes
- Log transform stabilizes variance and improves linear model performance
- `+1` handles zero-std hours (flat or single-interval hours)
- Still continuous → compatible with regression, GARCH, and ML models

A secondary **binary spike flag** (`rtm_price_mean > $100` or `rtm_price_std > $50`) can complement the regression target for classification tasks.

In [ ]:
# --- Target variable distribution analysis ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

_PROC = Path('data/processed/ercot')
rtm = pd.read_parquet(_PROC / 'np6_905_rtm_hourly_houston.parquet')[['ts_utc', 'rtm_price_std', 'rtm_price_mean']]
rtm = rtm.dropna(subset=['rtm_price_std'])

log_std = np.log1p(rtm['rtm_price_std'])
spike_flag = (rtm['rtm_price_mean'] > 100).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Raw rtm_price_std distribution
axes[0].hist(rtm['rtm_price_std'], bins=100, color='steelblue', edgecolor='none')
axes[0].set_xlabel('rtm_price_std ($/MWh)')
axes[0].set_title('Raw rtm_price_std\n(right-skewed)')
axes[0].set_yscale('log')

# 2. Log-transformed distribution
axes[1].hist(log_std, bins=100, color='seagreen', edgecolor='none')
axes[1].set_xlabel('log(rtm_price_std + 1)')
axes[1].set_title('Log-transformed target\n(more symmetric)')

# 3. Spike flag class balance
spike_counts = spike_flag.value_counts().sort_index()
axes[2].bar(['Normal\n(<=100)', 'Spike\n(>100)'], spike_counts.values, color=['steelblue', 'tomato'])
axes[2].set_title('Binary spike flag\nclass balance')
axes[2].set_ylabel('Hours')
for i, v in enumerate(spike_counts.values):
    axes[2].text(i, v + 50, f'{v:,}\n({v/len(rtm)*100:.1f}%)', ha='center', fontsize=9)

plt.suptitle('RTM Price Volatility — Target Variable Comparison', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"rtm_price_std stats:")
print(rtm['rtm_price_std'].describe().round(2))
print(f"\nlog(rtm_price_std+1) stats:")
print(log_std.describe().round(3))
print(f"\nSpike rate: {spike_flag.mean()*100:.2f}% of hours")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

_PROC = Path('data/processed/ercot')

# Load target
rtm = pd.read_parquet(_PROC / 'np6_905_rtm_hourly_houston.parquet')[['ts_utc', 'rtm_price_std', 'rtm_price_mean']]
rtm['log_rtm_std'] = np.log1p(rtm['rtm_price_std'])

# Load features
anc_regup = pd.read_parquet(_PROC / 'np4_188_mcpc_regup.parquet')[['ts_utc', 'mcpc_regup']]
anc_rrs   = pd.read_parquet(_PROC / 'np4_188_mcpc_rrs.parquet')[['ts_utc', 'mcpc_rrs']]
anc_ecrs  = pd.read_parquet(_PROC / 'np4_188_mcpc_ecrs.parquet')[['ts_utc', 'mcpc_ecrs']]
anc_nspin = pd.read_parquet(_PROC / 'np4_188_mcpc_nspin.parquet')[['ts_utc', 'mcpc_nspin']]
anc_regdn = pd.read_parquet(_PROC / 'np4_188_mcpc_regdn.parquet')[['ts_utc', 'mcpc_regdn']]
dam  = pd.read_parquet(_PROC / 'np4_190_dam_houston.parquet')[['ts_utc', 'dam_price_houston']]
lam  = pd.read_parquet(_PROC / 'np4_523_system_lambda.parquet')[['ts_utc', 'system_lambda']]
wind = pd.read_parquet(_PROC / 'np4_732_wind_system.parquet')[['ts_utc', 'wind_error_system', 'wgrpp_system_wide']]
out  = pd.read_parquet(_PROC / 'np3_233_outage_total.parquet')[['ts_utc', 'total_resource_mw']]
fc   = pd.read_parquet(_PROC / 'np3_565_forecast1.parquet')[['ts_utc', 'fc_system_total']]

df = rtm.copy()
for d in [anc_regup, anc_rrs, anc_ecrs, anc_nspin, anc_regdn, dam, lam, wind, out, fc]:
    df = df.merge(d, on='ts_utc', how='left')

df = df.sort_values('ts_utc').reset_index(drop=True)
df['rtm_mean_lag24'] = df['rtm_price_mean'].shift(24)

feat_cols = ['mcpc_regup', 'mcpc_rrs', 'mcpc_ecrs', 'mcpc_nspin', 'mcpc_regdn',
             'dam_price_houston', 'system_lambda', 'wind_error_system',
             'wgrpp_system_wide', 'total_resource_mw', 'fc_system_total', 'rtm_mean_lag24']

corr = df[feat_cols + ['log_rtm_std']].corr()['log_rtm_std'].drop('log_rtm_std').sort_values(key=abs, ascending=False)
print("Pearson correlation with log(rtm_price_std+1):")
print(corr.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['steelblue' if v > 0 else 'tomato' for v in corr.values]
ax.barh(corr.index[::-1], corr.values[::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson r with log(rtm_price_std + 1)')
ax.set_title('Feature Correlation with RTM Price Volatility')
plt.tight_layout()
plt.savefig('agent_notes/volatility_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.close()
print("Plot saved.")

### Non-Linear Effects: Wind Error × Load Interaction

Pearson correlations only capture linear relationships. Key non-linear hypotheses:

1. **Wind error only matters when load is high** — a 1 GW wind miss is benign at low demand but catastrophic near peak
2. **ECRS price has a threshold effect** — volatility jumps sharply once ECRS crosses a scarcity price level
3. **DAM price × outage interaction** — high outages amplify the effect of high DAM prices

We test these by:
- Binning `fc_system_total` into load quartiles and computing correlation of `wind_error_system` with `log_rtm_std` within each bin
- Scatter plots of key interactions with `log_rtm_std` as color/size
- Correlation of interaction terms (e.g. `wind_error × fc_system_total`) vs individual features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

_PROC = Path('data/processed/ercot')

# Load data
rtm  = pd.read_parquet(_PROC / 'np6_905_rtm_hourly_houston.parquet')[['ts_utc', 'rtm_price_std', 'rtm_price_mean']]
wind = pd.read_parquet(_PROC / 'np4_732_wind_system.parquet')[['ts_utc', 'wind_error_system', 'wgrpp_system_wide']]
fc   = pd.read_parquet(_PROC / 'np3_565_forecast1.parquet')[['ts_utc', 'fc_system_total']]
out  = pd.read_parquet(_PROC / 'np3_233_outage_total.parquet')[['ts_utc', 'total_resource_mw']]
dam  = pd.read_parquet(_PROC / 'np4_190_dam_houston.parquet')[['ts_utc', 'dam_price_houston']]
ecrs = pd.read_parquet(_PROC / 'np4_188_mcpc_ecrs.parquet')[['ts_utc', 'mcpc_ecrs']]

df = rtm.copy()
for d in [wind, fc, out, dam, ecrs]:
    df = df.merge(d, on='ts_utc', how='left')

df['log_rtm_std'] = np.log1p(df['rtm_price_std'])
df = df.dropna(subset=['log_rtm_std', 'fc_system_total', 'wind_error_system'])

# Interaction terms
df['wind_x_load'] = df['wind_error_system'] * df['fc_system_total']
df['dam_x_outage'] = df['dam_price_houston'] * df['total_resource_mw']
df['load_quartile'] = pd.qcut(df['fc_system_total'], q=4, labels=['Q1\n(low)', 'Q2', 'Q3', 'Q4\n(high)'])

# --- Plot ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Wind error correlation with log_rtm_std by load quartile
quartile_corrs = df.groupby('load_quartile', observed=True).apply(
    lambda g: g['wind_error_system'].corr(g['log_rtm_std'])
)
axes[0, 0].bar(quartile_corrs.index, quartile_corrs.values, color=['#a8d8ea', '#7bc8f6', '#3a86ff', '#023e8a'])
axes[0, 0].axhline(0, color='black', linewidth=0.8)
axes[0, 0].set_xlabel('Load quartile (fc_system_total)')
axes[0, 0].set_ylabel('Pearson r (wind_error vs log_rtm_std)')
axes[0, 0].set_title('Hypothesis 1: Wind error matters\nmore when load is high')

# 2. ECRS threshold effect — scatter with threshold line
ecrs_data = df.dropna(subset=['mcpc_ecrs'])
axes[0, 1].scatter(ecrs_data['mcpc_ecrs'].clip(upper=500), ecrs_data['log_rtm_std'],
                   alpha=0.05, s=3, color='steelblue')
axes[0, 1].axvline(50, color='tomato', linewidth=1.5, linestyle='--', label='$50 threshold')
axes[0, 1].set_xlabel('mcpc_ecrs ($/MWh, clipped at $500)')
axes[0, 1].set_ylabel('log(rtm_price_std + 1)')
axes[0, 1].set_title('Hypothesis 2: ECRS threshold effect')
axes[0, 1].legend()

# 3. Interaction term correlations vs individual
base_corrs = {
    'wind_error': df['wind_error_system'].corr(df['log_rtm_std']),
    'fc_system_total': df['fc_system_total'].corr(df['log_rtm_std']),
    'wind × load': df['wind_x_load'].corr(df['log_rtm_std']),
    'dam_price': df['dam_price_houston'].corr(df['log_rtm_std']),
    'outage_mw': df['total_resource_mw'].corr(df['log_rtm_std']),
    'dam × outage': df['dam_x_outage'].corr(df['log_rtm_std']),
}
colors = ['tomato' if '×' in k else 'steelblue' for k in base_corrs]
axes[1, 0].barh(list(base_corrs.keys()), list(base_corrs.values()), color=colors)
axes[1, 0].axvline(0, color='black', linewidth=0.8)
axes[1, 0].set_xlabel('Pearson r with log(rtm_price_std + 1)')
axes[1, 0].set_title('Hypothesis 3: Interaction terms\nvs individual features (red = interaction)')

# 4. Wind error vs log_rtm_std colored by load quartile
for q, color in zip(['Q1\n(low)', 'Q2', 'Q3', 'Q4\n(high)'], ['#a8d8ea', '#7bc8f6', '#3a86ff', '#023e8a']):
    mask = df['load_quartile'] == q
    axes[1, 1].scatter(df.loc[mask, 'wind_error_system'].clip(-3000, 3000),
                       df.loc[mask, 'log_rtm_std'],
                       alpha=0.1, s=2, color=color, label=q.replace('\n', ' '))
axes[1, 1].set_xlabel('wind_error_system (MW, clipped)')
axes[1, 1].set_ylabel('log(rtm_price_std + 1)')
axes[1, 1].set_title('Wind error vs volatility\nby load quartile')
axes[1, 1].legend(title='Load quartile', markerscale=4)

plt.suptitle('Non-Linear Feature Effects on RTM Price Volatility', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('agent_notes/volatility_nonlinear_effects.png', dpi=150, bbox_inches='tight')
plt.close()
print("Plot saved.")

# Print interaction term correlations
print("\nInteraction term correlations with log(rtm_price_std+1):")
for k, v in base_corrs.items():
    print(f"  {k:<22} r = {v:.3f}")

print("\nWind error correlation by load quartile:")
print(quartile_corrs.round(3).to_string())

### Non-Linear Effects — Conclusions

**Hypothesis 1 — Wind error × load**: NOT confirmed. Wind error is weakly negative at low load (r=−0.13) but near zero at peak. No load-quartile amplification. Drop `wind × load` interaction.

**Hypothesis 2 — ECRS threshold**: CONFIRMED. Volatility fans out sharply above ~$50 ECRS. Add:
- `log(mcpc_ecrs + 1)` — continuous transform
- `ecrs_above_50` — binary threshold flag

**Hypothesis 3 — DAM × outage interaction**: NOT confirmed. Interaction term (r=0.094) weaker than DAM alone (r=0.165). Drop.

---

### Final Feature Set for Modeling

| Feature | Transform | Source |
|---|---|---|
| `mcpc_ecrs` | `log(x+1)` + binary `ecrs_above_50` | np4_188 |
| `fc_system_total` | as-is | np3_565 |
| `dam_price_houston` | as-is | np4_190 |
| `system_lambda` | as-is | np4_523 |
| `mcpc_regup`, `mcpc_rrs`, `mcpc_nspin` | as-is | np4_188 |
| `total_resource_mw` (outages) | as-is | np3_233 |
| `rtm_mean_lag24` | as-is | np6_905 |
| `wgrpp_system_wide` | as-is | np4_732 |
| `wind_error_system` | tentative — weak signal | np4_732 |
| Weather features (temp, humidity) | TBD | weather_hourly (pending) |

**Target**: `log(rtm_price_std + 1)` (regression) · `rtm_price_mean > $100` (binary, secondary)

**Train window**: 2017-07 → 2024-12 · **Test**: 2025 · **Hold-out**: 2026

## Feature Matrix Audit Summary

Audited by EDAAgent — 2026-03-15. No direct data leakage found.

### Issues Fixed

| # | Severity | Issue | Fix |
|---|---|---|---|
| 1 | Medium | Outage fallback used `sort_values('ts_utc').tail(24)` — borrowed values from arbitrary past delivery slot | Changed to `sort_values('post_datetime').iloc[-1]` — uses most recently published outage value |
| 2 | Low | Raw `mcpc_ecrs` not filled — NaN for all pre-2021 rows | Added `fillna(0)` with comment explaining pre-2021 = ECRS not yet procured |
| 3 | Low | `ecrs_above_50=0` ambiguous (NaN pre-2021 vs genuine ≤$50) | Added `ecrs_available` flag (1 = ECRS procured that hour, 0 = pre-2021 or not procured) |

### Leakage Audit: All Clear

| Feature | Available at 6PM D-1? | Filter | Status |
|---|---|---|---|
| DAM price, system lambda, ancillary MCPC | Yes — published D-1 noon | `_avail()` + same-day D | ✅ |
| Load actuals, RTM prices, wind actuals | Yes — D-2 published D-1 morning | `_avail()` + d2_date + +2 day shift | ✅ |
| Outage capacity | Yes — hourly rolling, most recent post ≤ cutoff | `_avail()` + delivery day | ✅ |
| Load forecast, wind forecast | Yes — 6PM cutoff baked in via horizon_h | `_day()` only (no post_datetime) | ✅ |
| Target (`log_rtm_std`) | N/A — joined externally, not inside `build_features()` | External join only | ✅ |

### Training Matrix Statistics
- **Shape**: (65,712 rows × 26 columns) after fixes
- **Date range**: 2017-07-04 → 2024-12-31
- **Target nulls**: 0
- **Notable nulls**: `mcpc_ecrs` 80% null pre-2021 (now filled with 0); ancillary/DAM ~2,296 nulls each (DAM non-operating gaps)
- **Saved**: `data/processed/ercot/train_features.parquet` (7.2 MB)

### DST Note
Spring-forward on D-2 produces a NaN at the missing hour in D-2 lag features after the +2-day shift. This is correct behaviour — a genuine missing observation. Fall-back (25-hour D-2) is handled by existing deduplication.